# 商品阶段整理：从原表独立复现
你已在第 12 份 Notebook 检查商品、品类翻译、重量与尺寸，并把商品属性关联到销售明细。
本份将这些步骤整理为独立流程。原始 CSV 和前面的 Notebook 保留不变。

**本课目标：能说清每一步输入、输出和一行代表什么，并复现控制总额。**
这里只覆盖商品阶段；卖家、地理、MySQL 和 Power BI 继续按后续课程推进。

| 工作表 | 一行代表什么 | 主键 |
| --- | --- | --- |
| products_clean | 一个商品的属性 | product_id |
| items_with_products | 订单中的一条商品明细，并附上商品属性 | order_id + order_item_id |
| raw_translation | 一个葡萄牙语品类的英文映射 | product_category_name |

同一商品多次出现在销售明细中是正常业务记录。属性表中的 product_id 必须唯一，销售明细中的 product_id 可以重复。


## 1. 读三张原表，导入已经整理的函数
本课仍使用原始字符串读取，金额由此前的商品明细函数从原始字符串精确换成整数分。
函数实现集中在 src/products_cleaning.py；这里展示调用顺序，不要求你重新抄写整套函数。

先逐格运行。若修改过公共函数，请重启内核后从头运行，避免旧导入留在内存中。


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# 支持从项目根目录或 notebooks 目录打开本课。
project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从 olist-operations-analytics 项目目录打开 Notebook')

data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import clean_order_items
from products_cleaning import (
    NUMERIC_COLS, check_product_numeric, clean_product_attributes,
    join_product_categories, join_items_products,
    reconcile_product_totals, product_attribute_coverage, summarize_product_metrics
)

def read_source(filename):
    # dtype=str 使 ID 和原始金额保持字符串。
    # 只把空字段识别为缺失，不擅自把某个文本标签解释为 NA。
    return pd.read_csv(
        data_dir / filename, dtype=str, encoding='utf-8-sig',
        keep_default_na=False, na_values=['']
    )

raw_products = read_source('olist_products_dataset.csv')
raw_translation = read_source('product_category_name_translation.csv')
raw_items = read_source('olist_order_items_dataset.csv')

print('商品表行数：', len(raw_products))
print('品类翻译表行数：', len(raw_translation))
print('商品明细表行数：', len(raw_items))


商品表行数： 32951
品类翻译表行数： 71
商品明细表行数： 112650


## 2. 商品关联品类翻译，分别检查七个数值属性
join_product_categories 返回商品表的工作副本，加入英文品类和两个缺失原因标记。
它先检查商品 ID、映射键非空且唯一，再按品类名做 many_to_one 左连接，输出仍一行一个商品。

两类英文名缺失：原始品类缺失、原始品类有值但未匹配翻译。它们都保留。
check_product_numeric 只做诊断，不会改写输入；“原始缺失”与“非空但解析失败”需要区分。


In [2]:
products_with_categories = join_product_categories(raw_products, raw_translation)

# 显示两类缺失原因，不把 623 个商品全部说成翻译表缺漏。
display(pd.Series({
    '关联后商品数': len(products_with_categories),
    '原始品类缺失商品数': products_with_categories['flag_category_missing'].sum(),
    '有原始品类但无翻译商品数': products_with_categories['flag_category_untranslated'].sum(),
    '英文品类缺失商品数': products_with_categories['product_category_name_english'].isna().sum()
}))

# 检查表本身不转换原表；下一格才生成规范类型后的商品工作表。
numeric_check = check_product_numeric(products_with_categories)
display(numeric_check)


关联后商品数          32951
原始品类缺失商品数         610
有原始品类但无翻译商品数       13
英文品类缺失商品数         623
dtype: int64

,原始缺失数,非空但解析失败数,非有限数,非整数数,负数数,零值数
product_name_lenght,610,0,0,0,0,0
product_description_lenght,610,0,0,0,0,0
product_photos_qty,610,0,0,0,0,0
product_weight_g,2,0,0,0,0,4
product_length_cm,2,0,0,0,0,0
product_height_cm,2,0,0,0,0,0
product_width_cm,2,0,0,0,0,0


## 3. 规范属性类型，按各指标的条件计算特征
clean_product_attributes 返回完整副本，保留全部 32,951 个商品。当前来源的七个数值属性采用可空整数 Int64，未知仍为空。

- can_use_weight：重量非空且大于 0。
- can_calculate_volume：长、宽、高都非空且大于 0。
- product_volume_cm3：满足体积条件时计算三边乘积，否则为空。

这些条件互相独立。缺少重量、品类或图片信息，不会自动取消完整尺寸的体积资格。
非空解析失败、非有限值、小数、负数会停止并要求核查，不进行舍入或猜测修复。


In [3]:
products_clean = clean_product_attributes(products_with_categories)

# 返回值仍然是一整张商品表，不是只有 flag 或体积的一小段结果。
display(products_clean[NUMERIC_COLS + ['product_volume_cm3']].dtypes)

# fillna(False) 这里只是为了选出“已知重量为 0”的样本，
# 不会把原表的未知重量或未知 flag 改成 0 / False。
zero_weight_sample = products_clean.loc[
    products_clean['flag_weight_zero'].fillna(False),
    ['product_id', 'product_weight_g', 'can_use_weight',
     'can_calculate_volume', 'product_volume_cm3']
]
display(zero_weight_sample)

print('可计算体积商品数：', products_clean['can_calculate_volume'].sum())
print('体积未知商品数：', products_clean['product_volume_cm3'].isna().sum())


product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
product_weight_g              Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
product_volume_cm3            Int64
dtype: object

,product_id,product_weight_g,can_use_weight,can_calculate_volume,product_volume_cm3
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,0,False,True,22500
13683,8038040ee2a71048d4bdbbdc985b69ab,0,False,True,22500
14997,36ba42dd187055e1fbe943b2d11430ca,0,False,True,22500
32079,e673e90efa65a5409ff4196c038bb5af,0,False,True,22500


可计算体积商品数： 32949
体积未知商品数： 2


## 4. 为销售明细补商品属性
clean_order_items 复用前面完成的金额与明细主键规范。join_items_products 再以商品 ID 关联属性。
左侧同一商品可被多次购买，右侧商品 ID 必须唯一，因此验证 many_to_one。

**这一步没有按 product_id 去重，也没有按 order_id 汇总。**
输出仍是一行一条商品明细；订单中的不同明细序号需要保留。


In [4]:
items_clean = clean_order_items(raw_items)
items_with_products = join_items_products(items_clean, products_clean)

# 核对明细行数和组合主键；不能改成检查 product_id 是否唯一。
print('关联前明细行数：', len(items_clean))
print('关联后明细行数：', len(items_with_products))
print('关联后组合主键重复数：',
      items_with_products.duplicated(['order_id', 'order_item_id']).sum())

# both 表示找到商品记录；不表示该商品的每一项属性都有值。
display(items_with_products['products_match'].value_counts())


关联前明细行数： 112650
关联后明细行数： 112650
关联后组合主键重复数： 0


products_match
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

## 5. 核对数量、商品金额和运费
属性关联不应该改变来源销售事实。比较前后明细行数、整数分商品金额及运费。
三个差额都为 0，且关联后的明细组合键唯一，才允许导出；异常时函数会停止。

本格使用全部明细、全部订单状态。这里的商品金额不同于之前筛选 delivered 后的商品成交额。


In [5]:
product_join_check = reconcile_product_totals(items_clean, items_with_products)
display(product_join_check)

# 预期：明细数量 112650，商品金额 1359164370 分，运费 225190954 分；
# 每行的“关联前”和“关联后”相同，差额都是 0。
# 这张控制表有 3 行，是 3 个核对项目，不是只有 3 笔订单。


,关联前,关联后,差额
商品明细行数,112650,112650,0
商品金额（分）,1359164370,1359164370,0
运费（分）,225190954,225190954,0


## 6. 分开看“商品数”和“销售明细数”
一个缺少属性的商品可能销售多次，因此还需要知道受影响的销售明细数。
这里不删除这些记录，只统计影响范围。不同质量条件可能重叠，不能相加当作去重后的异常总数。


In [6]:
attribute_coverage = product_attribute_coverage(products_clean, items_with_products)
display(attribute_coverage)

# .loc 只筛选出“有原始品类但无翻译”的商品。
# groupby + size 才将这些商品按原始品类汇总为一行一个品类。
untranslated_categories = (
    products_clean.loc[products_clean['flag_category_untranslated']]
    .groupby('product_category_name')
    .size()
    .rename('商品数')
    .reset_index()
)
display(untranslated_categories)


,商品数,销售明细数
原始品类缺失,610,1603
有品类但无翻译,13,24
重量不满足基础条件,6,26
体积不可计算,2,18


,product_category_name,商品数
0,pc_gamer,3
1,portateis_cozinha_e_preparadores_de_alimentos,10


## 7. 汇总本阶段结果
metrics 是便于后续脚本记录的字典；它不会替代完整商品表或销售明细表。
这里的体积是记录尺寸的乘积。没有额外业务核查时，不称为真实包装体积或计费体积。


In [7]:
metrics = summarize_product_metrics(
    products_clean, raw_translation, items_with_products
)

display(pd.Series({
    '商品数': metrics['product_rows'],
    '销售明细数': metrics['item_rows'],
    '涉及订单数': metrics['item_orders'],
    '涉及不同商品数': metrics['item_product_ids'],
    '无商品匹配明细数': metrics['items_without_product'],
    '可计算体积商品数': metrics['volume_eligible_products']
}))

# describe 的 count 是参与体积统计的有效商品数，不是所有商品数。
display(
    products_clean['product_volume_cm3']
    .describe(percentiles=[0.5, 0.9, 0.99])
    .round(2)
)


商品数          32951
销售明细数       112650
涉及订单数        98666
涉及不同商品数      32951
无商品匹配明细数         0
可计算体积商品数     32949
dtype: int64

count      32949.0
mean       16564.1
std       27057.04
min          168.0
50%         6840.0
90%        41238.8
99%      135917.44
max       296208.0
Name: product_volume_cm3, dtype: Float64

## 导出与本课练习
本 Notebook 只在内存中复现检查。需要更新本阶段独立导出时，在项目目录的终端运行：

    python .\src\07_clean_products.py

数据在 data/processed/products_v1；检查表和清单在 reports/products_v1。
CSV 重新读取时需按字段恢复 ID 字符串、可空整数、布尔和日期，不能依赖自动类型推断。

**我的理解（在下面补充）：**

1. products_clean 和 items_with_products 的主键分别是什么？为什么不能按 product_id 删除重复销售明细？
products_clean的主键是product_id,items_with_products的主键是order_id + order_item_id 。
2. 原始品类缺失有 610 个商品，却涉及 1,603 条销售明细，为什么这不是数量核对失败？
一个缺少属性的商品可能销售多次。
3. clean_product_attributes 的输入、返回值分别是什么？缺少重量的商品是否一定不能计算体积？
输入是和翻译表左连接之后的商品表，返回值是完整副本，包含规范后的数值属性、核查标记和体积，缺少重量不一定不能计算体积。

下次先核对你的解释，再进入卖家表。
